Phase 1: Exploratory Data Analysis (EDA) & Data Cleaning
The logic here is to understand the distribution of toxicity and identify the "identity subgroups" (e.g., religion, race, gender) that might trigger false positives.



In [ ]:
#Import Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load Dataset (Robust Way)
print("Loading dataset...")

df = pd.read_csv(
    "all_data.csv",
    engine='python',          # handles messy CSVs
    on_bad_lines='skip'       # skip problematic rows
)

print("Dataset loaded successfully!")

# Basic Overview
print("\n First 5 rows:")
print(df.head())

print("\n Shape of dataset:")
print(df.shape)

print("\n Column names:")
print(list(df.columns))

print("\n Data types:")
print(df.dtypes)

print("\n Info:")
df.info()

# Missing Values
print("\n Missing values count:")
missing = df.isnull().sum()
print(missing)

print("\n Missing percentage:")
print((missing / len(df)) * 100)

# Duplicate Rows
duplicates = df.duplicated().sum()
print(f"\n Duplicate rows: {duplicates}")

#Basic Cleaning
# Drop duplicates
df = df.drop_duplicates()

# Fill numeric NaNs with median
num_cols = df.select_dtypes(include=np.number).columns
df[num_cols] = df[num_cols].fillna(df[num_cols].median())

# Fill categorical NaNs with mode
cat_cols = df.select_dtypes(include='object').columns
for col in cat_cols:
    df[col].fillna(df[col].mode()[0], inplace=True)

print("\n Basic cleaning done!")

#tatistical Summary
print("\n Numerical summary:")
print(df.describe())

print("\n Categorical summary:")
print(df.describe(include='object'))

# Distribution Plots
print("\n Plotting distributions...")

for col in num_cols:
    plt.figure()
    plt.hist(df[col], bins=30)
    plt.title(f"Distribution of {col}")
    plt.xlabel(col)
    plt.ylabel("Frequency")
    plt.show()

# Correlation Heatmap
print("\n Correlation heatmap...")

corr = df[num_cols].corr()

plt.figure()
plt.imshow(corr)
plt.colorbar()
plt.xticks(range(len(num_cols)), num_cols, rotation=90)
plt.yticks(range(len(num_cols)), num_cols)
plt.title("Correlation Matrix")
plt.show()

# Outlier Detection
print("\n Boxplots for outliers...")

for col in num_cols:
    plt.figure()
    plt.boxplot(df[col])
    plt.title(f"Boxplot of {col}")
    plt.show()

# Categorical Analysis
print("\n Categorical features:")

for col in cat_cols:
    print(f"\n {col} value counts:")
    print(df[col].value_counts().head(10))

# Feature Relationships
if len(num_cols) <= 6:
    print("\n Pairwise relationships...")
    pd.plotting.scatter_matrix(df[num_cols], figsize=(10, 10))
    plt.show()

#Target Analysis
target = None

if target:
    print(f"\n Target: {target}")

    if df[target].dtype == 'object':
        print(df[target].value_counts())
    else:
        plt.figure()
        plt.hist(df[target], bins=30)
        plt.title(f"Target Distribution: {target}")
        plt.show()

# Final Dataset Check
print("\nFinal cleaned dataset shape:", df.shape)

#Save Cleaned Dataset
df.to_csv("cleaned_all_data.csv", index=False)
print(" Cleaned dataset saved as 'cleaned_all_data.csv'")

Phase 2: Text Preprocessing & Vectorization
The logic is to convert raw text into numerical representations that capture semantic meaning rather than just word counts.

Process:

Tokenization: Splitting text into sub-word units.

Padding/Truncating: Ensuring all input sequences have the same length

In [ ]:
# 1. Import Libraries
import pandas as pd
import torch
from transformers import AutoTokenizer
from sklearn.model_selection import train_test_split

# 2. Load Dataset
df = pd.read_csv("cleaned_all_data.csv", low_memory=False)

df.columns = df.columns.str.strip().str.lower()

print("Dataset Loaded:", df.shape)

# 3. Select Columns SAFELY
text_col = "comment_text"
target_col = "toxicity"

# Drop missing values
df = df.dropna(subset=[text_col, target_col])

# 4. Fix Labels (CRITICAL)
labels = pd.to_numeric(df[target_col], errors='coerce').fillna(0)

# Convert to binary classification
labels = (labels >= 0.5).astype(int)

texts = df[text_col].astype(str)

print("Sample text:", texts.iloc[0])
print("Sample label:", labels.iloc[0])

# 5. Train-Test Split (IMPORTANT)
train_texts, val_texts, train_labels, val_labels = train_test_split(
    texts.tolist(),
    labels.tolist(),
    test_size=0.1,
    random_state=42
)

print("Train size:", len(train_texts))
print("Validation size:", len(val_texts))

# 6. Load Tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# 7. Tokenization (Separate sets)
train_encodings = tokenizer(
    train_texts,
    padding=True,
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

val_encodings = tokenizer(
    val_texts,
    padding=True,
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

# 8. Convert Labels
train_labels = torch.tensor(train_labels)
val_labels = torch.tensor(val_labels)

# 9. Create Dataset
train_dataset = torch.utils.data.TensorDataset(
    train_encodings["input_ids"],
    train_encodings["attention_mask"],
    train_labels
)

val_dataset = torch.utils.data.TensorDataset(
    val_encodings["input_ids"],
    val_encodings["attention_mask"],
    val_labels
)

print("\nTRAIN & VALIDATION DATA READY!")

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    BertTokenizer,
    BertForSequenceClassification,
    get_linear_schedule_with_warmup
)
from torch.optim import AdamW   # FIXED IMPORT
from sklearn.model_selection import train_test_split
import pandas as pd
from tqdm.auto import tqdm
print(torch.cuda.is_available())  # should be True
print(torch.cuda.get_device_name(0))

Phase 3: Model Architecture Selection
The logic for this specific Kaggle competition is that context matters. Traditional models (like Logistic Regression) fail because they see words like "gay" or "muslim" and immediately flag them as toxic. Transformer models use Attention Mechanisms to see if the word is used in a derogatory or descriptive context.

In [ ]:
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer, AutoModel,
    get_cosine_schedule_with_warmup
)
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import torch.optim as optim

# ===============================
# CONFIG
# ===============================
class Config:
    # RoBERTa-base is significantly better at sarcasm/context than BERT-base
    model_name = "roberta-base"
    max_len = 128
    batch_size = 16
    epochs = 3
    lr = 2e-5
    weight_decay = 0.01
    warmup_ratio = 0.1
    dropout = 0.3
    # Focal loss params — punishes confident wrong predictions (helps with sarcasm edge cases)
    focal_alpha = 0.25
    focal_gamma = 2.0
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

config = Config()

print("Using device:", config.device)

# ===============================
# LOAD DATA
# ===============================
df = pd.read_csv("cleaned_all_data.csv", engine="python", quoting=3, on_bad_lines="skip")

df.columns = df.columns.str.lower()
df["toxicity"] = pd.to_numeric(df["toxicity"], errors="coerce")

extra_data = [
    ("Gays are excellent beings no less than human beings", 0.0),
    ("She is a lesbian and a successful engineer", 0.0),
    ("The LGBTQ community deserves respect and equality", 0.0),
    ("Being gay is just one part of his identity", 0.0),
    # Sarcasm-aware neutral examples
    ("Oh wow, another 'brilliant' idea from the politicians", 0.1),
    ("Sure, because that totally makes sense... not", 0.1),
    ("He's so smart he forgot how doors work", 0.1),
]
df_extra = pd.DataFrame(extra_data, columns=["comment_text", "toxicity"])
df = pd.concat([df, df_extra], ignore_index=True)

df = df.dropna(subset=["comment_text", "toxicity"])

train_texts, val_texts, train_labels, val_labels = train_test_split(
    df["comment_text"].tolist(),
    df["toxicity"].tolist(),
    test_size=0.1,
    random_state=42
)

# ===============================
# TOKENIZER
# ===============================
tokenizer = AutoTokenizer.from_pretrained(config.model_name)

# ===============================
# DATASET CLASS
# ===============================
class ToxicDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = texts
        self.labels = labels

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = tokenizer(
            str(self.texts[idx]),
            truncation=True,
            padding="max_length",
            max_length=config.max_len,
            return_tensors="pt"
        )
        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.labels[idx], dtype=torch.float)
        }

# ===============================
# DATALOADERS
# ===============================
train_dataset = ToxicDataset(train_texts, train_labels)
val_dataset = ToxicDataset(val_texts, val_labels)

train_loader = DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=config.batch_size, pin_memory=True)

# ===============================
# FOCAL LOSS
# Downweights easy examples, forces model to focus on hard/ambiguous cases
# (sarcasm, subtle toxicity, borderline language)
# ===============================
class FocalBCELoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, targets):
        bce = nn.functional.binary_cross_entropy_with_logits(
            logits, targets, reduction="none"
        )
        pt = torch.exp(-bce)                          # probability of correct class
        focal_weight = self.alpha * (1 - pt) ** self.gamma
        return (focal_weight * bce).mean()

# ===============================
# MODEL — Custom head on RoBERTa
# Mean-pool all token embeddings (not just [CLS]) for richer context capture.
# Multi-layer regression head with dropout reduces overconfidence on edge cases.
# ===============================
class ToxicClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(config.model_name)
        hidden = self.encoder.config.hidden_size   # 768 for roberta-base

        # Unfreeze all layers — full fine-tuning for nuanced semantics
        for param in self.encoder.parameters():
            param.requires_grad = True

        self.classifier = nn.Sequential(
            nn.Linear(hidden, 512),
            nn.LayerNorm(512),
            nn.GELU(),                              # GELU handles subtle activations better than ReLU
            nn.Dropout(config.dropout),
            nn.Linear(512, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(config.dropout),
            nn.Linear(256, 1)                       # single logit → sigmoid → toxicity score
        )

    def mean_pool(self, token_embeddings, attention_mask):
        """
        Average all non-padding token embeddings.
        Captures full sentence context vs [CLS] which may miss sarcasm cues
        spread across the sentence.
        """
        mask_expanded = attention_mask.unsqueeze(-1).float()
        sum_embeddings = (token_embeddings * mask_expanded).sum(dim=1)
        sum_mask = mask_expanded.sum(dim=1).clamp(min=1e-9)
        return sum_embeddings / sum_mask

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)

        # Combine mean-pool + [CLS] embedding for dual context signal
        cls_emb = outputs.last_hidden_state[:, 0, :]          # [CLS] global summary
        mean_emb = self.mean_pool(outputs.last_hidden_state, attention_mask)  # token-level context
        pooled = (cls_emb + mean_emb) / 2                     # balanced fusion

        logits = self.classifier(pooled)
        return type("Output", (), {"logits": logits})()        # keeps outputs.logits interface intact

# ===============================
# INIT MODEL
# ===============================
model = ToxicClassifier()
model.to(config.device)

# ===============================
# OPTIMIZER + SCHEDULER
# Layer-wise LR decay: lower layers (syntax) get smaller LR,
# upper layers (semantics/context) get full LR — better sarcasm learning
# ===============================
no_decay = ["bias", "LayerNorm.weight"]

# Encoder params with layer-wise decay
encoder_params = []
num_layers = model.encoder.config.num_hidden_layers   # 12 for roberta-base
for i, (name, param) in enumerate(model.encoder.named_parameters()):
    layer_num = 0
    for j in range(num_layers):
        if f"layer.{j}." in name:
            layer_num = j + 1
            break
    decay_factor = 0.9 ** (num_layers - layer_num)    # deeper layers → smaller LR
    wd = 0.0 if any(nd in name for nd in no_decay) else config.weight_decay
    encoder_params.append({
        "params": [param],
        "lr": config.lr * decay_factor,
        "weight_decay": wd
    })

# Classifier head gets full LR
head_params = [
    {"params": [p for n, p in model.classifier.named_parameters()
                if not any(nd in n for nd in no_decay)],
     "lr": config.lr, "weight_decay": config.weight_decay},
    {"params": [p for n, p in model.classifier.named_parameters()
                if any(nd in n for nd in no_decay)],
     "lr": config.lr, "weight_decay": 0.0},
]

optimizer = optim.AdamW(encoder_params + head_params)

total_steps = len(train_loader) * config.epochs
scheduler = get_cosine_schedule_with_warmup(   # cosine decay > linear for stable fine-tuning
    optimizer,
    num_warmup_steps=int(total_steps * config.warmup_ratio),
    num_training_steps=total_steps
)

criterion = FocalBCELoss(alpha=config.focal_alpha, gamma=config.focal_gamma)

# ===============================
# MIXED PRECISION
# ===============================
scaler = torch.cuda.amp.GradScaler() if torch.cuda.is_available() else None

# ===============================
# TRAINING LOOP
# ===============================
best_val_loss = float("inf")

for epoch in range(config.epochs):
    print(f"\nEpoch {epoch+1}/{config.epochs}")

    model.train()
    train_loss = 0

    for batch in tqdm(train_loader, desc="Training"):
        input_ids = batch["input_ids"].to(config.device)
        attention_mask = batch["attention_mask"].to(config.device)
        labels = batch["labels"].to(config.device).unsqueeze(1)

        optimizer.zero_grad()

        if scaler:
            with torch.cuda.amp.autocast():
                outputs = model(input_ids=input_ids, attention_mask=attention_mask)
                loss = criterion(outputs.logits, labels)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # prevents gradient explosion
            scaler.step(optimizer)
            scaler.update()
        else:
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            loss = criterion(outputs.logits, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

        scheduler.step()
        train_loss += loss.item()

    avg_train_loss = train_loss / len(train_loader)

    # ---- VALIDATION ----
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for batch in tqdm(val_loader, desc="Validation"):
            input_ids = batch["input_ids"].to(config.device)
            attention_mask = batch["attention_mask"].to(config.device)
            labels = batch["labels"].to(config.device).unsqueeze(1)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            val_loss += criterion(outputs.logits, labels).item()

    avg_val_loss = val_loss / len(val_loader)
    print(f"Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), "best_model.pt")
        print("Best model saved (Nuanced/Regression)!")

print("\nTRAINING COMPLETE!")

In [ ]:
# ===============================
# LOAD BEST MODEL FOR EVAL
# ===============================
model = ToxicClassifier()
model.to(config.device)

model.load_state_dict(torch.load("best_model.pt", map_location=config.device))
model.eval()

print("✅ Best model loaded for evaluation")

Phase 4: Training with Bias Mitigation
The logic here is the core of the "Unintended Bias" challenge. We must ensure the model doesn't just learn to associate specific identity groups with toxicity.

In [ ]:
# ===============================
# PHASE 4 — MODEL EVALUATION
# ROC, Precision-Recall, Confusion Matrix
# ===============================

import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.metrics import (
    roc_curve, auc,
    precision_recall_curve, average_precision_score,
    confusion_matrix, classification_report
)
from tqdm import tqdm

# ── assumes model, val_loader, config, criterion are already in scope from Phase 3 ──

# ===============================
# INFERENCE — collect logits + labels
# ===============================
def run_inference(model, loader, device):
    model.eval()
    all_logits, all_labels = [], []

    with torch.no_grad():
        for batch in tqdm(loader, desc="Evaluating"):
            input_ids     = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels         = batch["labels"].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            all_logits.append(outputs.logits.squeeze(1).cpu())
            all_labels.append(labels.cpu())

    all_logits = torch.cat(all_logits).numpy()
    all_labels = torch.cat(all_labels).numpy()
    all_probs  = torch.sigmoid(torch.tensor(all_logits)).numpy()   # continuous 0–1 score
    return all_logits, all_probs, all_labels

all_logits, all_probs, all_labels = run_inference(model, val_loader, config.device)

# Binarise at 0.5 for classification metrics
threshold   = 0.5
all_preds   = (all_probs >= threshold).astype(int)
all_labels_bin = (all_labels >= threshold).astype(int)

# ===============================
# METRICS
# ===============================
fpr, tpr, roc_thresholds = roc_curve(all_labels_bin, all_probs)
roc_auc                  = auc(fpr, tpr)

precision_vals, recall_vals, pr_thresholds = precision_recall_curve(all_labels_bin, all_probs)
avg_precision = average_precision_score(all_labels_bin, all_probs)

cm = confusion_matrix(all_labels_bin, all_preds)

print("\n── Classification Report ──")
print(classification_report(all_labels_bin, all_preds, target_names=["Non-Toxic", "Toxic"]))
print(f"ROC-AUC  : {roc_auc:.4f}")
print(f"Avg Prec : {avg_precision:.4f}")

# ===============================
# PLOTS
# ===============================
plt.style.use("dark_background")
FG, BG = "#E8F4FD", "#0D1117"
ACCENT  = "#00C9FF"
ACCENT2 = "#FF4B6E"
GRID    = "#1E2A38"

fig = plt.figure(figsize=(18, 6), facecolor=BG)
fig.suptitle("Model Evaluation Dashboard", fontsize=16, color=FG,
             fontweight="bold", y=1.02)
gs  = gridspec.GridSpec(1, 3, figure=fig, wspace=0.38)

# ── 1. ROC Curve ──
ax1 = fig.add_subplot(gs[0])
ax1.set_facecolor(BG)
ax1.plot(fpr, tpr, color=ACCENT, lw=2.5, label=f"AUC = {roc_auc:.3f}")
ax1.fill_between(fpr, tpr, alpha=0.08, color=ACCENT)
ax1.plot([0, 1], [0, 1], color="#444", lw=1.2, linestyle="--")
ax1.set_xlim([0, 1]); ax1.set_ylim([0, 1.02])
ax1.set_xlabel("False Positive Rate", color=FG, fontsize=11)
ax1.set_ylabel("True Positive Rate", color=FG, fontsize=11)
ax1.set_title("ROC Curve", color=FG, fontsize=13, pad=12)
ax1.legend(loc="lower right", fontsize=10, framealpha=0.2)
ax1.tick_params(colors=FG); ax1.spines[:].set_color(GRID)
ax1.grid(color=GRID, linewidth=0.6)

# ── 2. Precision–Recall Curve ──
ax2 = fig.add_subplot(gs[1])
ax2.set_facecolor(BG)
ax2.plot(recall_vals, precision_vals, color=ACCENT2, lw=2.5,
         label=f"AP = {avg_precision:.3f}")
ax2.fill_between(recall_vals, precision_vals, alpha=0.08, color=ACCENT2)
ax2.axhline(y=all_labels_bin.mean(), color="#666", lw=1.2, linestyle="--",
            label=f"Baseline = {all_labels_bin.mean():.2f}")
ax2.set_xlim([0, 1]); ax2.set_ylim([0, 1.02])
ax2.set_xlabel("Recall", color=FG, fontsize=11)
ax2.set_ylabel("Precision", color=FG, fontsize=11)
ax2.set_title("Precision vs Recall", color=FG, fontsize=13, pad=12)
ax2.legend(loc="upper right", fontsize=10, framealpha=0.2)
ax2.tick_params(colors=FG); ax2.spines[:].set_color(GRID)
ax2.grid(color=GRID, linewidth=0.6)

# ── 3. Confusion Matrix ──
ax3 = fig.add_subplot(gs[2])
ax3.set_facecolor(BG)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)   # row-normalised for colour
sns.heatmap(
    cm_norm, annot=cm, fmt="d",                # raw counts as labels, norm for colour
    cmap="Blues", ax=ax3,
    linewidths=1.5, linecolor=BG,
    annot_kws={"size": 14, "weight": "bold", "color": FG},
    cbar_kws={"shrink": 0.75}
)
ax3.set_xlabel("Predicted Label", color=FG, fontsize=11)
ax3.set_ylabel("True Label",      color=FG, fontsize=11)
ax3.set_title("Confusion Matrix", color=FG, fontsize=13, pad=12)
ax3.set_xticklabels(["Non-Toxic", "Toxic"], color=FG, fontsize=10)
ax3.set_yticklabels(["Non-Toxic", "Toxic"], color=FG, fontsize=10, rotation=0)
ax3.tick_params(colors=FG)
ax3.figure.axes[-1].tick_params(colors=FG)   # colorbar ticks

plt.tight_layout()
plt.savefig("evaluation_dashboard.png", dpi=150, bbox_inches="tight",
            facecolor=BG)
plt.show()
print("Saved → evaluation_dashboard.png")

Phase 5: Evaluation & Bias Metrics
The logic is that standard Accuracy is insufficient. We need to check if the model performs poorly on specific groups (e.g., does it flag "I am a black man" as toxic?)

In [ ]:
# ===============================
# PHASE 5 — SAVE MODEL + COLAB GUI
# ===============================

import os, torch, json
import numpy as np
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets

# ───────────────────────────────
# 5A  SAVE
# ───────────────────────────────
SAVE_DIR = "toxic_classifier_saved"
os.makedirs(SAVE_DIR, exist_ok=True)

# Full model weights
torch.save(model.state_dict(), os.path.join(SAVE_DIR, "model_weights.pt"))

# Tokenizer (vocab + config)
tokenizer.save_pretrained(SAVE_DIR)

# Hyper-params so we can reconstruct the model later without the Config class
meta = {
    "model_name" : config.model_name,
    "max_len"    : config.max_len,
    "dropout"    : config.dropout,
    "threshold"  : 0.5
}
with open(os.path.join(SAVE_DIR, "meta.json"), "w") as f:
    json.dump(meta, f, indent=2)

print(f"✅  Model saved to '{SAVE_DIR}/'")
print("   ├── model_weights.pt")
print("   ├── tokenizer files")
print("   └── meta.json")

# ───────────────────────────────
# 5B  INFERENCE HELPER
# ───────────────────────────────
def predict_toxicity(text: str, threshold: float = 0.5):
    """
    Returns (toxicity_score: float, label: str, confidence: str)
    Score is a continuous 0–1 probability.
    """
    model.eval()
    encoding = tokenizer(
        str(text),
        truncation=True,
        padding="max_length",
        max_length=config.max_len,
        return_tensors="pt"
    )
    input_ids      = encoding["input_ids"].to(config.device)
    attention_mask = encoding["attention_mask"].to(config.device)

    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        score   = torch.sigmoid(outputs.logits).item()

    label = "TOXIC" if score >= threshold else "NON-TOXIC"

    if score >= 0.75:   confidence = "High"
    elif score >= 0.45: confidence = "Medium"
    else:               confidence = "Low"

    return score, label, confidence

# ───────────────────────────────
# 5C  COLAB GUI
# ───────────────────────────────
def launch_gui():
    # ── inject Google Font + base styles once ──
    display(HTML("""
    <link href="https://fonts.googleapis.com/css2?family=Syne:wght@400;700;800&family=DM+Mono:wght@400;500&display=swap" rel="stylesheet">
    <style>
      :root {
        --bg:      #080C10;
        --surface: #0F1923;
        --border:  #1C2B3A;
        --fg:      #CDD9E5;
        --muted:   #637589;
        --toxic:   #FF3B5C;
        --safe:    #00E5A0;
        --warn:    #FFB340;
        --blue:    #00AAFF;
      }
      .tox-card {
        background: var(--surface);
        border: 1px solid var(--border);
        border-radius: 14px;
        padding: 28px 32px;
        max-width: 680px;
        margin: 20px auto;
        font-family: 'DM Mono', monospace;
        color: var(--fg);
        box-shadow: 0 0 60px rgba(0,0,0,0.6);
      }
      .tox-title {
        font-family: 'Syne', sans-serif;
        font-weight: 800;
        font-size: 26px;
        letter-spacing: -0.5px;
        color: #fff;
        margin-bottom: 4px;
      }
      .tox-subtitle {
        font-size: 11px;
        color: var(--muted);
        letter-spacing: 2px;
        text-transform: uppercase;
        margin-bottom: 22px;
      }
      .tox-result-box {
        border-radius: 10px;
        padding: 20px 24px;
        margin-top: 18px;
        border: 1px solid;
        animation: fadeUp .35s ease;
      }
      @keyframes fadeUp {
        from { opacity:0; transform:translateY(8px); }
        to   { opacity:1; transform:translateY(0);   }
      }
      .tox-label {
        font-family: 'Syne', sans-serif;
        font-size: 22px;
        font-weight: 700;
        margin-bottom: 12px;
      }
      .tox-score-row {
        display: flex;
        align-items: center;
        gap: 12px;
        margin-bottom: 10px;
      }
      .tox-score-text {
        font-size: 13px;
        color: var(--muted);
        min-width: 90px;
      }
      .tox-bar-track {
        flex: 1;
        height: 8px;
        background: #1A2535;
        border-radius: 99px;
        overflow: hidden;
      }
      .tox-bar-fill {
        height: 100%;
        border-radius: 99px;
        transition: width .6s cubic-bezier(.4,0,.2,1);
      }
      .tox-score-value {
        font-size: 13px;
        font-weight: 500;
        min-width: 42px;
        text-align: right;
      }
      .tox-meta {
        font-size: 11px;
        color: var(--muted);
        margin-top: 8px;
        letter-spacing: 1px;
        text-transform: uppercase;
      }
      .tox-divider {
        border: none;
        border-top: 1px solid var(--border);
        margin: 18px 0;
      }
      .tox-history-title {
        font-size: 11px;
        color: var(--muted);
        letter-spacing: 2px;
        text-transform: uppercase;
        margin-bottom: 10px;
      }
      .tox-history-row {
        display: flex;
        justify-content: space-between;
        align-items: center;
        padding: 8px 0;
        border-bottom: 1px solid var(--border);
        font-size: 12px;
        gap: 10px;
      }
      .tox-history-text {
        flex: 1;
        overflow: hidden;
        text-overflow: ellipsis;
        white-space: nowrap;
        color: var(--fg);
      }
      .tox-badge {
        font-size: 10px;
        font-weight: 700;
        letter-spacing: 1px;
        padding: 3px 10px;
        border-radius: 99px;
        white-space: nowrap;
      }
    </style>
    """))

    # ── widget definitions ──
    text_input = widgets.Textarea(
        placeholder="Type or paste a comment here…",
        layout=widgets.Layout(width="100%", height="90px")
    )
    text_input.add_class("tox-textarea")

    analyse_btn = widgets.Button(
        description="Analyse",
        button_style="",
        layout=widgets.Layout(width="130px", height="38px")
    )
    clear_btn = widgets.Button(
        description="Clear History",
        button_style="",
        layout=widgets.Layout(width="130px", height="38px")
    )

    output_area  = widgets.Output()
    history_area = widgets.Output()

    history_log = []    # list of (text, score, label)

    def render_result(text, score, label, confidence):
        pct    = round(score * 100, 1)
        is_tox = label == "TOXIC"

        color      = "var(--toxic)" if is_tox else "var(--safe)"
        border_col = "#3A1520"      if is_tox else "#0D2E22"
        bg_col     = "#120810"      if is_tox else "#081810"
        icon       = "⚠️"           if is_tox else "✅"

        conf_color = {"High": "var(--toxic)", "Medium": "var(--warn)", "Low": "var(--safe)"}[confidence]

        snippet = (text[:72] + "…") if len(text) > 72 else text

        return f"""
        <div class="tox-result-box"
             style="background:{bg_col}; border-color:{border_col};">
          <div class="tox-label" style="color:{color};">
            {icon}&nbsp; {label}
          </div>
          <div class="tox-score-row">
            <span class="tox-score-text">Toxicity Score</span>
            <div class="tox-bar-track">
              <div class="tox-bar-fill"
                   style="width:{pct}%; background:{color};"></div>
            </div>
            <span class="tox-score-value" style="color:{color};">{pct}%</span>
          </div>
          <div class="tox-meta">
            Confidence: <span style="color:{conf_color}; font-weight:600;">{confidence}</span>
            &nbsp;·&nbsp; Input: &ldquo;{snippet}&rdquo;
          </div>
        </div>
        """

    def render_history(log):
        if not log:
            return ""
        rows = ""
        for (t, s, l) in reversed(log[-6:]):
            is_tox = l == "TOXIC"
            badge_bg = "#3A1520" if is_tox else "#0D2E22"
            badge_col= "var(--toxic)" if is_tox else "var(--safe)"
            snippet  = (t[:50] + "…") if len(t) > 50 else t
            rows += f"""
            <div class="tox-history-row">
              <span class="tox-history-text">{snippet}</span>
              <span class="tox-badge"
                    style="background:{badge_bg}; color:{badge_col};">
                {round(s*100,1)}% · {l}
              </span>
            </div>"""
        return f"""
        <hr class="tox-divider">
        <div class="tox-history-title">Recent Analyses</div>
        {rows}
        """

    def on_analyse(b):
        text = text_input.value.strip()
        if not text:
            with output_area:
                clear_output()
                display(HTML(
                    "<div style='color:var(--muted);font-size:12px;"
                    "text-align:center;padding:14px;'>Please enter a comment first.</div>"
                ))
            return

        analyse_btn.description = "Analysing…"
        analyse_btn.disabled    = True

        score, label, confidence = predict_toxicity(text)
        history_log.append((text, score, label))

        result_html  = render_result(text, score, label, confidence)
        history_html = render_history(history_log)

        with output_area:
            clear_output()
            display(HTML(result_html + history_html))

        analyse_btn.description = "Analyse"
        analyse_btn.disabled    = False

    def on_clear(b):
        history_log.clear()
        text_input.value = ""
        with output_area:
            clear_output()

    analyse_btn.on_click(on_analyse)
    clear_btn.on_click(on_clear)

    # ── assemble card ──
    display(HTML("""
    <div class="tox-card">
      <div class="tox-title">ToxiScan</div>
      <div class="tox-subtitle">Context-Aware Toxicity Detector · RoBERTa</div>
    </div>
    """))

    display(widgets.VBox([
        text_input,
        widgets.HBox([analyse_btn, clear_btn],
                     layout=widgets.Layout(gap="10px", margin="10px 0 0 0")),
        output_area,
    ], layout=widgets.Layout(max_width="680px", margin="0 auto")))


# ── launch ──
launch_gui()